# Phase 3: Real Point Cloud Processing & OptiTrack Alignment

This notebook prepares raw physical scan data (`view*.pcd`) for surface analysis and PointNet benchmarking:
1. **RANSAC Plane Segmentation**: Cleans table/ground plane noise from physical scans.
2. **Oriented Bounding Box (OBB) Cropping**: Isolates the workpiece ROI.
3. **OptiTrack Ground Truth Alignment**: Uses the physical OptiTrack pose with manual marker-to-CAD offset ($T_{base \to CAD} = T_{optitrack} \cdot T_{offset}$).
4. **Export for Surface Analysis**: Saves cleaned scans into `processed_data/` so `6_surface_analysis.ipynb` can compute per-surface Chamfer Distances.

## 1. Helper Functions

In [2]:
import os
import re
import copy
import glob
import math
import numpy as np
import open3d as o3d
from scipy.spatial.transform import Rotation as R
import pandas as pd

def get_rotation_matrix_z(deg):
    """Creates a 3x3 rotation matrix for the Z-axis."""
    rad = math.radians(deg)
    c, s = math.cos(rad), math.sin(rad)
    return np.array([[c, -s, 0],
                     [s,  c, 0],
                     [0,  0, 1]])

def pose_to_matrix(pose):
    """
    Converts [x, y, z, roll, pitch, yaw] (angles in degrees) to a 4x4 transformation matrix.
    """
    x, y, z = pose[:3]
    roll, pitch, yaw = pose[3:]
    rot_matrix = R.from_euler('xyz', [roll, pitch, yaw], degrees=True).as_matrix()
    T = np.eye(4)
    T[:3, :3] = rot_matrix
    T[:3, 3] = [x, y, z]
    return T

def clean_and_crop_point_cloud(pcd, initial_pos, initial_angle, box_size, remove_plane=True, distance_threshold=3.0, plane_offset=0.5):
    """
    Processes a single raw physical PCD: removes ground plane and crops to OBB.
    """
    if pcd.is_empty():
        return pcd
    
    pcd_clean = copy.deepcopy(pcd)
    
    # 1. Detect and remove table/ground plane using RANSAC
    if remove_plane:
        plane_model, inliers = pcd_clean.segment_plane(distance_threshold=distance_threshold, ransac_n=3, num_iterations=2000)
        [a, b, c, d] = plane_model
        pts = np.asarray(pcd_clean.points)
        distances = a * pts[:, 0] + b * pts[:, 1] + c * pts[:, 2] + d
        above_plane_indices = np.where(distances > plane_offset)[0]
        pcd_clean = pcd_clean.select_by_index(above_plane_indices)
        
    # 2. Crop to Oriented Bounding Box around the workpiece
    rot_matrix = get_rotation_matrix_z(-initial_angle)
    obb = o3d.geometry.OrientedBoundingBox(
        center=np.array(initial_pos),
        R=rot_matrix,
        extent=np.array(box_size)
    )
    final_pcd = pcd_clean.crop(obb)
    return final_pcd

def extract_number(filename):
    """Extracts numerical ID from filename (e.g. view12.pcd -> 12)."""
    numbers = re.findall(r'\d+', os.path.basename(filename))
    return int(numbers[0]) if numbers else 0


Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


## 2. Interactive Visual Validation & Manual Offset Adjustment
Set your OptiTrack matrix and manual offsets below ($T_{base \to CAD} = T_{optitrack} \cdot T_{offset}$).

- `OFFSET_TRANSLATION_MM`: `[dx, dy, dz]` offset in mm from OptiTrack marker pivot to CAD origin.
- `OFFSET_ROTATION_DEG`: `[roll, pitch, yaw]` rotation adjustment in degrees.

In [ ]:
# ==========================================
# 1. CONFIGURATION: EXPERIMENT & WORKPIECE
# ==========================================
EXPERIMENT = "test_10_realgrasp"
WORKPIECE = "workpiece31"
VIEWPOINT_IDX = 231

DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}"
CAD_PATH = f"workpiece/{WORKPIECE}/workpiece.stl"

# Load Initial YOLO / Rough Object Pose for Cropping Box
yolo_pose_path = os.path.join(DATA_DIR, "initial_obj_pose.npy")
if os.path.exists(yolo_pose_path):
    tf_obj = np.load(yolo_pose_path)
    YOLO_POS = tf_obj[:3].copy()
    YOLO_ANGLE = float(tf_obj[4])
    YOLO_POS[0] -= 0  # Visual crop adjustment if needed
    YOLO_POS[1] -= -10
else:
    YOLO_POS = [540.0, -70.0, 20.0]  # Fallback approximate center (mm)
    YOLO_ANGLE = 0.0

# Define Bounding Box Size (mm) around workpiece: [X_extent, Y_extent, Z_extent]
CROP_BOX = [80, 105, 70]  # Adjust ROI dimensions for your workpiece

# ==========================================
# 2. OPTITRACK RAW MATRIX (Translation in mm)
# ==========================================
T_optitrack = np.array([
    [ 9.999e-01,  5.000e-04, -1.380e-02,  514.6],
    [-4.000e-04,  1.000e+00,  2.300e-03, -104.3],
    [ 1.380e-02, -2.300e-03,  9.999e-01,   -8.6],
    [ 0.000e+00,  0.000e+00,  0.000e+00,    1.0]
])

# ==========================================
# 3. MANUAL OFFSET (MARKER PIVOT -> CAD ORIGIN)
# ==========================================
# Translation offset in mm: [dx, dy, dz]
OFFSET_TRANSLATION_MM = [-4,4,6] # [0.0, 0.0, 4] # + 4

# Rotation offset in degrees: [roll, pitch, yaw] (XYZ Euler)
OFFSET_ROTATION_DEG = [0.0, 0.0, 90.0]

# Compute Final Ground Truth Pose: T_gt = T_optitrack @ T_offset
T_offset = np.eye(4)
T_offset[:3, :3] = R.from_euler('xyz', OFFSET_ROTATION_DEG, degrees=True).as_matrix()
T_offset[:3, 3] = OFFSET_TRANSLATION_MM

T_ground_truth = T_optitrack @ T_offset

print("Final Ground Truth Transformation Matrix (T_ground_truth):")
print(T_ground_truth)
print(f"  - Position (mm): {np.round(T_ground_truth[:3, 3], 2)}")

# ==========================================
# 4. LOAD CAD & REAL SCAN (SINGLE VIEW)
# ==========================================
mesh = o3d.io.read_triangle_mesh(CAD_PATH)
mesh.compute_vertex_normals()
cad_cloud = mesh.sample_points_uniformly(number_of_points=40000)

pcd_file = os.path.join(DATA_DIR, f"view{VIEWPOINT_IDX}.pcd")
if not os.path.exists(pcd_file):
    pcd_file = os.path.join(DATA_DIR, f"view{VIEWPOINT_IDX:02d}.pcd")

raw_pcd = o3d.io.read_point_cloud(pcd_file)

# Create Oriented Bounding Box for Visualization
rot_matrix = get_rotation_matrix_z(-YOLO_ANGLE)
obb = o3d.geometry.OrientedBoundingBox(
    center=np.array(YOLO_POS),
    R=rot_matrix,
    extent=np.array(CROP_BOX)
)
obb.color = (0.0, 1.0, 0.0)  # Green wireframe box

# Clean table plane and crop to OBB
clean_pcd = clean_and_crop_point_cloud(raw_pcd, YOLO_POS, YOLO_ANGLE, CROP_BOX, remove_plane=True)

# ==========================================
# 5. VISUAL VERIFICATION & CROP BOX INSPECTION
# ==========================================
cad_aligned = copy.deepcopy(cad_cloud).transform(T_ground_truth)
cad_aligned.paint_uniform_color([1.0, 0.0, 0.0])       # Red: CAD Ground Truth
clean_pcd.paint_uniform_color([0.0, 0.65, 0.93])      # Blue: Real Scanned Point Cloud
raw_pcd_vis = copy.deepcopy(raw_pcd).paint_uniform_color([0.6, 0.6, 0.6]) # Grey: Raw Uncropped

world_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=50.0, origin=T_ground_truth[:3, 3])
crop_frame = o3d.geometry.TriangleMesh.create_coordinate_frame(size=30.0, origin=YOLO_POS)

print(f"\n1. Window 1: RAW Scan (Grey) + GREEN Cropping Box...")
print(f"   -> Verify the green box fits around the workpiece before closing window.")
o3d.visualization.draw_geometries([raw_pcd_vis, obb], window_name="1. Raw Scan & Cropping Box (Green Wireframe)")

print(f"\n2. Window 2: CROPPED Scan (Blue) + CAD Model (Red) + Cropping Box (Green)...")
o3d.visualization.draw_geometries([clean_pcd, cad_aligned, obb, world_frame], window_name=f"2. OptiTrack Alignment & Crop - View {VIEWPOINT_IDX}")


Final Ground Truth Transformation Matrix (T_ground_truth):
[[ 5.000000e-04 -9.999000e-01 -1.380000e-02  5.105196e+02]
 [ 1.000000e+00  4.000000e-04  2.300000e-03 -1.002846e+02]
 [-2.300000e-03 -1.380000e-02  9.999000e-01 -2.665000e+00]
 [ 0.000000e+00  0.000000e+00  0.000000e+00  1.000000e+00]]
  - Position (mm): [ 510.52 -100.28   -2.66]

1. Window 1: RAW Scan (Grey) + GREEN Cropping Box...
   -> Verify the green box fits around the workpiece before closing window.

2. Window 2: CROPPED Scan (Blue) + CAD Model (Red) + Cropping Box (Green)...


: 

## 3. Batch Processing Loop (Exporting for Surface Analysis)
This cell cleans and processes all real viewpoint scans, transforms them to the CAD coordinate frame using the manual OptiTrack ground truth pose, and exports them directly into `processed_data/` so that **`6_surface_analysis.ipynb`** can segment the surface features and generate `metadata.csv`.

In [10]:
# ==========================================
# BATCH REAL DATA PROCESSING PIPELINE
# ==========================================
EXPERIMENT = "test_10_realgrasp"
WORKPIECES = ["workpiece31"]  # Add all workpieces to process

for workpiece in WORKPIECES:
    print(f"\n========================================")
    print(f"PROCESSING REAL WORKPIECE: {workpiece}")
    print(f"========================================")
    
    DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}"
    if not os.path.exists(DATA_DIR):
        DATA_DIR = f"pcd_data/testing_data/{EXPERIMENT}/{workpiece}"
        
    CAD_PATH = f"workpiece/{workpiece}/workpiece.stl"
    PROCESSED_DIR = f"processed_data/{EXPERIMENT}/{workpiece}"
    EVAL_DIR = f"evaluation_result/{EXPERIMENT}/{workpiece}"
    os.makedirs(PROCESSED_DIR, exist_ok=True)
    os.makedirs(EVAL_DIR, exist_ok=True)
    
    # 1. Compute and Save Ground Truth Matrix
    T_gt = T_optitrack @ T_offset
    np.save(os.path.join(EVAL_DIR, "merge_full_transformation.npy"), T_gt)
    print(f"Saved Ground Truth transformation to: {EVAL_DIR}/merge_full_transformation.npy")
    
    # 2. Find all real scan files (e.g. view*.pcd)
    pcd_files = [f for f in os.listdir(DATA_DIR) if f.startswith("view") and f.endswith(".pcd") and "_surface" not in f]
    pcd_files.sort(key=extract_number)
    
    print(f"Found {len(pcd_files)} real scan files. Processing and aligning to CAD frame...")
    
    T_inv_gt = np.linalg.inv(T_gt)  # Transform real points into CAD coordinate frame
    
    exported_count = 0
    for file_name in pcd_files:
        view_idx = extract_number(file_name)
        file_path = os.path.join(DATA_DIR, file_name)
        raw_pcd = o3d.io.read_point_cloud(file_path)
        
        # Clean ground plane & Crop workpiece ROI
        clean_pcd = clean_and_crop_point_cloud(raw_pcd, YOLO_POS, YOLO_ANGLE, CROP_BOX, remove_plane=True)
        
        if len(clean_pcd.points) < 50:
            print(f"  [Warning] Viewpoint {view_idx} has too few points ({len(clean_pcd.points)}), skipping.")
            continue
            
        # Transform scan directly into CAD coordinate frame for 6_surface_analysis
        clean_pcd.transform(T_inv_gt)
        
        # Export with standard naming format recognized by 6_surface_analysis.ipynb
        export_filename = f"viewpoint_simulated_noise_{view_idx}.pcd"
        export_path = os.path.join(PROCESSED_DIR, export_filename)
        o3d.io.write_point_cloud(export_path, clean_pcd)
        exported_count += 1
        
    print(f"[SUCCESS] Exported {exported_count} cleaned & CAD-aligned PCDs to: {PROCESSED_DIR}")

print("\n==========================================")
print("ALL REAL DATA PROCESSED!")
print("You can now run 6_surface_analysis.ipynb to extract surface features & compute Chamfer errors.")
print("==========================================")



PROCESSING REAL WORKPIECE: workpiece31
Saved Ground Truth transformation to: evaluation_result/test_10_realgrasp/workpiece31/merge_full_transformation.npy
Found 287 real scan files. Processing and aligning to CAD frame...
[SUCCESS] Exported 287 cleaned & CAD-aligned PCDs to: processed_data/test_10_realgrasp/workpiece31

ALL REAL DATA PROCESSED!
You can now run 6_surface_analysis.ipynb to extract surface features & compute Chamfer errors.


In [1]:
print("a")

a
